# Vlnová rovnice: Minkowski vs. hyperboloidální řezy

Vizualizace řešení sféricky symetrické vlnové rovnice pro **odcházející gaussovský balík**, dvakrát:

1. **Minkowski** — standardní řezy $T = \mathrm{konst}$, $R \in [0, 10]$, na vnějším okraji je nutná Sommerfeldova podmínka.
2. **Hyperboloidální řezy + kompaktifikace** — $r \in [0, r_{\mathcal{I}}]$; budoucí nulové nekonečno $\mathcal{I}^+$ leží přímo na mřížce a signál na něm čteme bez jakékoli okrajové podmínky.

In [ ]:
import numpy as np

# RK integrátor a derivace

In [ ]:
RK_COEFFICIENT_TABLES = {
    1: ([0], [[0]], [1]),
    2: ([0, 1 / 2], [[0, 0], [1 / 2, 0]], [0, 1]),
    3: ([0, 1 / 2, 1], [[0, 0, 0], [1 / 2, 0, 0], [-1, 2, 0]], [1 / 6, 2 / 3, 1 / 6]),
    4: (
        [0, 1 / 2, 1 / 2, 1],
        [[0, 0, 0, 0], [1 / 2, 0, 0, 0], [0, 1 / 2, 0, 0], [0, 0, 1, 0]],
        [1 / 6, 1 / 3, 1 / 3, 1 / 6],
    ),
    5: (
        [0, 1 / 5, 3 / 10, 4 / 5, 8 / 9, 1, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 5, 0, 0, 0, 0, 0, 0],
            [3 / 40, 9 / 40, 0, 0, 0, 0, 0],
            [44 / 45, -56 / 15, 32 / 9, 0, 0, 0, 0],
            [19372 / 6561, -25360 / 2187, 64448 / 6561, -212 / 729, 0, 0, 0],
            [9017 / 3168, -355 / 33, 46732 / 5247, 49 / 176, -5103 / 18656, 0, 0],
            [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
        ],
        [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84, 0],
    ),
    6: (
        [0, 1 / 3, 2 / 3, 1 / 3, 1 / 2, 1 / 2, 1],
        [
            [0, 0, 0, 0, 0, 0, 0],
            [1 / 3, 0, 0, 0, 0, 0, 0],
            [0, 2 / 3, 0, 0, 0, 0, 0],
            [1 / 12, 1 / 3, -1 / 12, 0, 0, 0, 0],
            [-1 / 16, 9 / 8, -3 / 16, -3 / 8, 0, 0, 0],
            [0, 9 / 8, -3 / 8, -3 / 4, 1 / 2, 0, 0],
            [9 / 44, -9 / 11, 63 / 44, 18 / 11, 0, -16 / 11, 0],
        ],
        [11 / 120, 0, 27 / 40, 27 / 40, -4 / 15, -4 / 15, 11 / 120],
    ),
}


In [ ]:
class RKp:
    def __init__(self, order: int = 4):
        """
        Args:
            t0 (float): Starting time
            h (float): Time step
            dydt (function): function in shape  dy/dt = f(t, y)
            tmax (float, optional): Stopping time of integration. Defaults to None.
            order (int, optional): Order of RK method used. Currently implemented are {1,2,3,4,5}. Defaults to 4.
        """

        self.order = order

        # load RK coefficients
        (c, A, b) = RK_COEFFICIENT_TABLES[order]
        (self.c, self.A, self.b) = (np.array(c), np.array(A), np.array(b))

    # initialization of values before integration process begins
    def Initialize(self, y0: np.ndarray, dydt):
        if len(y0) % 2 == 1:
            raise Exception("Wrong length of initial vector")
        self.y = y0.copy()
        self.ICS = y0.copy()
        self.y_history = [y0.copy()]
        self.dydt = dydt

    # integrate the given system
    def Integrate(self, t0: float, h: float, tmax: float = None):
        self.t0 = t0
        self.t = t0
        self.h = h

        while True:
            if self.t + h < tmax:
                self.NextStep()
            else:
                if tmax - self.t > 1e-14:
                    self.h = tmax - self.t
                    self.NextStep()
                break

    # calculate ks for general Butcher table
    def _getKs(self):
        k = np.zeros((len(self.c),) + np.shape(self.y))
        for i in range(len(self.c)):
            t = self.t + self.h * self.c[i]

            dy = np.zeros_like(self.y)
            for j in range(i):
                dy += self.A[i, j] * k[j]
            y = self.y + self.h * dy

            k[i] = self.dydt(t, y)
        return k.copy()

    # perform one step of RKp method
    def NextStep(self):
        # obtain k values
        k = self._getKs()
        # update y and t values
        self.y += self.h * sum(self.b[i] * k[i] for i in range(len(self.c)))
        self.t += self.h
        # save y value
        self.y_history.append(self.y.copy())
        return (self.t - self.t0) // self.h

    def GetHistory(self):
        return self.y_history

    @classmethod
    def GetAllImplemented(cls) -> dict:
        return RK_COEFFICIENT_TABLES.keys()


In [ ]:
def deriv_r_o4(arr, dx):
    res = np.zeros_like(arr)
    n = len(arr)

    if n < 5:
        raise ValueError("Pole musí mít alespoň 5 bodů pro 4. řád přesnosti.")

    # Central difference
    res[2:-2] = (-arr[4:] + 8 * arr[3:-1] - 8 * arr[1:-3] + arr[:-4]) / (12 * dx)

    # Borders with 4th order treatment
    res[0] = (-25*arr[0] + 48*arr[1] - 36*arr[2] + 16*arr[3] - 3*arr[4]) / (12 * dx)
    res[1] = (-3*arr[0] - 10*arr[1] + 18*arr[2] - 6*arr[3] + arr[4]) / (12 * dx)

    res[-1] = (25*arr[-1] - 48*arr[-2] + 36*arr[-3] - 16*arr[-4] + 3*arr[-5]) / (12 * dx)
    res[-2] = (3*arr[-1] + 10*arr[-2] - 18*arr[-3] + 6*arr[-4] - arr[-5]) / (12 * dx)

    return res

# Část 1: Minkowski

Integrujeme systém ($F = 0$):

$$
    \partial_T \psi = -\pi,
$$
$$
  \partial_T \phi = -\partial_R \pi + \gamma_2 (\partial_R\psi - \phi),
$$
$$
    \partial_T \pi = -\partial_R\phi - \frac{2}{R}\phi .
$$

Počáteční data volíme **čistě odcházející**: pro $\psi = f(T-R)/R$ platí $\pi = \phi + \psi/R$.

Vnější okraj $R_\mathrm{max}$ je umělý — bez Sommerfeldovy podmínky
$\partial_T \psi + \partial_R \psi + \psi/R = 0$ by se vlna odrážela zpět do domény.

In [ ]:
# --- Parametry gridu ---
dr = 0.1
r_min, r_max = 0, 10.0
r = np.arange(r_min, r_max + dr, dr)
N = len(r)

# --- Parametry pulsu ---
r0 = 4.0  # Střed pulsu
sigma = 0.5  # Šířka pulsu

# --- Parametry integrace ---
tmin = 0.0
timestep = 0.005
tmax = 15.0


In [ ]:
def system_rhs(r, dx, gamma2):
  def _system(t, y):
      """
      d psi /dt = - pi
      d phi /dt = - d pi/dr + gamma2 (d psi/dr - phi)
      d pi /dt = - d phi/dr - 2/R phi
      + Sommerfeldova odchozí podmínka na pravém okraji
      """
      # 1. Rozbalení vektoru y na jednotlivá pole
      N = len(y) // 3
      psi = y[0:N]
      phi = y[N : 2 * N]
      pi = y[2 * N : 3 * N]

      # 2. Inicializace derivací (rhs)
      dpsi_dt = np.zeros(N)
      dphi_dt = np.zeros(N)
      dpi_dt = np.zeros(N)

      # --- d psi /dt = -pi ---
      dpsi_dt[:] = - pi


      # --- d phi /dt = - d pi/dr + gamma2 (d psi/dr - phi)  - ---
      dphi_dt[1:] = - deriv_r_o4(pi, dx)[1:] + gamma2 * (deriv_r_o4(psi, dx) - phi)[1:]
      # Sudost Psi v r zaručuje v počátku Phi (= dPsi/dr) = 0
      dphi_dt[0] = 0.0

      # --- d pi /dt = - d phi/dr - 2/R phi ---
      dphi_dr = deriv_r_o4(phi, dx)
      dpi_dt[1:] = - dphi_dr[1:] - 2 * phi[1:]/r[1:]
      # Pro Pi l'Hospitalem dostaneme podmínku v počátku
      dpi_dt[0] = - 3 * dphi_dr[0]


      # --- Sommerfeld (odchozí vlna) ---
      # d_T psi + d_R psi + psi/R = 0, tj. pi = phi + psi/R;
      # derivací podle času: d_T pi = d_T phi + (d_T psi)/R = d_T phi - pi/R
      dpi_dt[-1] = dphi_dt[-1] - pi[-1] / r[-1]

      # 3. Zabalení zpět do jednoho vektoru
      return np.concatenate([dpsi_dt, dphi_dt, dpi_dt])
  return _system


In [ ]:
# --- Výpočet počátečních polí ---
# f = exp(-(r-r0)^2 / sigma^2)
psi = np.exp(-np.power(r - r0, 2) / np.power(sigma, 2))

phi = deriv_r_o4(psi, dr)

# Čistě odcházející balík: pi = phi + psi/R
# (v počátku je balík ~ e^{-64}, bezpečně nulujeme)
pi = np.zeros(N)
pi[1:] = phi[1:] + psi[1:] / r[1:]

# --- Zabalení pro třídu RKp ---
y0 = np.concatenate([psi, phi, pi])

if len(y0) % 2 != 0:
    # Pokud je délka lichá, upravíme grid o jeden bod, aby Initialize nehodilo chybu
    r = r[:-1]
    psi = psi[:-1]
    phi = phi[:-1]
    pi = pi[:-1]
    y0 = np.concatenate([psi, phi, pi])
    N = len(r)

print(f"Počáteční data připravena. Celková délka vektoru y0: {len(y0)} (3x {N})")

# Máme 4. řád derivace, proto raději 6. řád RK
solver = RKp(order=6)
solver.Initialize(y0, system_rhs(r, dr, 0.1))
solver.Integrate(t0=tmin, h=timestep, tmax=tmax)

print(f"Integrace ukončena")

history_mink = np.array(solver.GetHistory())[:, 0:N]  # pouze psi
print(f"max|psi| na konci: {np.abs(history_mink[-1]).max():.3e}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation


# --- ANIMACE: Minkowski ---
every_nth_frame = 30

fig, ax = plt.subplots(figsize=(10, 5))
(line,) = ax.plot(r, history_mink[0, :], lw=2, color="firebrick")

ax.set_xlim(r_min, r_max)
ax.set_ylim(-1, 1)
ax.set_title("Minkowski: odcházející balík $\\psi(T, R)$")
ax.set_xlabel("$R$")
ax.set_ylabel("$\\psi$")
ax.grid(True, linestyle="--", alpha=0.6)


def update(frame):
    line.set_ydata(history_mink[frame * every_nth_frame, :])
    return (line,)


ani = FuncAnimation(fig, update, frames=len(history_mink) // every_nth_frame, interval=60, blit=True, cache_frame_data=False)

plt.close()

from IPython.display import HTML
HTML(ani.to_jshtml())


# Část 2: Hyperboloidální řezy + kompaktifikace

**Transformace souřadnic** (výškovou funkcí a kompaktifikací, $S = r_{\mathcal{I}}$):

$$
t = T - h(R), \qquad h(R) = \sqrt{S^2 + R^2} - S, \qquad
R = \frac{r}{\Omega}, \qquad \Omega = 1 - \frac{r^2}{S^2}, \qquad r \in [0, S].
$$

Řezy $t = \mathrm{konst}$ jsou hyperboloidy: prostorupodobné všude, ale asymptoticky se přimykají
k nulovým kuželům, takže $\mathcal{I}^+$ odpovídá konečnému bodu $r = S$.

**Přeškálování.** Členy $\sim 1/R$ jsou po kompaktifikaci singulární a amplituda $\psi$ klesá jako $1/R$.
Obojí vyřeší proměnná $\chi = R\,\psi$, která splňuje čistou 1D vlnovou rovnici
$\partial_T^2 \chi = \partial_R^2 \chi$ a na $\mathcal{I}^+$ je přímo (přeškálovaným) radiačním signálem.

**Systém v proměnných** $\pi = -\partial_t \chi$, $\phi = \partial_r \chi$:

$$
\partial_t \chi = -\pi,
$$
$$
\partial_t \phi = -\partial_r \pi + \gamma_2 (\partial_r \chi - \phi),
$$
$$
\partial_t \pi = -\frac{1}{K}\Big[\, 2A\,\partial_r \pi + A'\,\pi + L^{-1}\partial_r \phi + \big(L^{-1}\big)'\,\phi \,\Big],
$$

kde ($'$ značí $\partial_r$, $Q = \sqrt{S^2\Omega^2 + r^2}$):

$$
A = h'\!\big(R(r)\big) = \frac{r}{Q}, \qquad
L^{-1} = \frac{\mathrm{d}r}{\mathrm{d}R} = \frac{\Omega^2}{1 + r^2/S^2}, \qquad
K = L\,(1 - A^2) = \frac{(1 + r^2/S^2)\,S^2}{Q^2}.
$$

Všechny koeficienty jsou **regulární na celé mřížce včetně $\mathcal{I}^+$**: tam $A = 1$, $L^{-1} = 0$, $K = 2$
a rovnice pro $\pi$ degeneruje na čistou advekci ven, $\partial_t \pi = -\partial_r \pi$.

**Charakteristické rychlosti**
$$
c_+ = \frac{1+A}{K}, \qquad c_- = -\frac{1-A}{K};
$$
v počátku $c_\pm = \pm 1$, na $\mathcal{I}^+$ je $c_+ = 1$ a $c_- = 0$ — dovnitř nic nepřichází,
takže na rozdíl od části 1 **žádnou okrajovou podmínku nepředepisujeme** (čistý outflow).

**Počáteční data**: čistě odcházející balík splňuje $\partial_T \chi = -\partial_R \chi$, což v nových
proměnných dává $\pi = \frac{1+A}{K}\,\phi$.

In [ ]:
# --- Kompaktifikace, výšková funkce a koeficienty systému ---
S = 1.0    # poloha scri v kompaktifikované souřadnici
N_h = 202  # sudý počet bodů kvůli kontrole v RKp.Initialize
rh = np.linspace(0.0, S, N_h)
dxh = rh[1] - rh[0]

Omega = 1 - (rh / S) ** 2
Q = np.sqrt(S**2 * Omega**2 + rh**2)
A = rh / Q                            # A(0) = 0, A(scri) = 1 přesně
invL = Omega**2 / (1 + (rh / S) ** 2) # 1/L, na scri 0
K = (1 + (rh / S) ** 2) * S**2 / Q**2 # K(0) = 1, K(scri) = 2

# Derivace koeficientů (hladké funkce, stačí numericky)
dA_dr = deriv_r_o4(A, dxh)
dinvL_dr = deriv_r_o4(invL, dxh)

# --- Charakteristické rychlosti ---
c_plus = (1 + A) / K
c_minus = -(1 - A) / K
print(f"max |c+| = {np.abs(c_plus).max():.3f}, max |c-| = {np.abs(c_minus).max():.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rh, c_plus, label="$c_+$ (ven)", color="firebrick")
ax.plot(rh, c_minus, label="$c_-$ (dovnitř)", color="steelblue")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("$r$")
ax.set_ylabel("rychlost")
ax.set_title("Charakteristické rychlosti: na $\\mathcal{I}^+$ je $c_-=0$")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
def system_rhs_hyp(dx, gamma2, A, invL, K, dA_dr, dinvL_dr):
  def _system(t, y):
      """
      d chi /dt = - pi
      d phi /dt = - d pi/dr + gamma2 (d chi/dr - phi)
      d pi /dt = - [ 2A d pi/dr + A' pi + (1/L) d phi/dr + (1/L)' phi ] / K
      Bez okrajové podmínky na scri (c_- = 0, čistý outflow).
      """
      N = len(y) // 3
      chi = y[0:N]
      phi = y[N : 2 * N]
      pi = y[2 * N : 3 * N]

      dchi_dt = np.zeros(N)
      dphi_dt = np.zeros(N)
      dpi_dt = np.zeros(N)

      # --- d chi /dt = -pi ---
      dchi_dt[:] = - pi

      # --- d phi /dt = - d pi/dr + gamma2 (d chi/dr - phi) ---
      dphi_dt[:] = - deriv_r_o4(pi, dx) + gamma2 * (deriv_r_o4(chi, dx) - phi)

      # --- d pi /dt ---
      dpi_dr = deriv_r_o4(pi, dx)
      dphi_dr = deriv_r_o4(phi, dx)
      dpi_dt[:] = - (2 * A * dpi_dr + dA_dr * pi + invL * dphi_dr + dinvL_dr * phi) / K

      # chi = R psi je lichá v R, tedy chi(0) = 0 a pi(0) = 0 (phi je sudá, tu necháme)
      dchi_dt[0] = 0.0
      dpi_dt[0] = 0.0

      return np.concatenate([dchi_dt, dphi_dt, dpi_dt])
  return _system


In [ ]:
# --- Parametry pulsu a integrace (hyperboloidální část) ---
rh0 = 0.4       # střed balíku v kompaktifikované souřadnici
sigma_h = 0.05  # šířka balíku
timestep_h = 0.001
tmax_h = 2.0

# --- Počáteční data: čistě odcházející balík ---
chi = np.exp(-np.power(rh - rh0, 2) / np.power(sigma_h, 2))
phi_h = deriv_r_o4(chi, dxh)
pi_h = (1 + A) * phi_h / K   # podmínka odchodnosti: pi = (1+A)/K * phi

y0_h = np.concatenate([chi, phi_h, pi_h])
print(f"Počáteční data připravena. Celková délka vektoru y0: {len(y0_h)} (3x {N_h})")

solver_h = RKp(order=6)
solver_h.Initialize(y0_h, system_rhs_hyp(dxh, 0.1, A, invL, K, dA_dr, dinvL_dr))
solver_h.Integrate(t0=0.0, h=timestep_h, tmax=tmax_h)

print(f"Integrace ukončena")

history_hyp = np.array(solver_h.GetHistory())[:, 0:N_h]  # pouze chi
print(f"max|chi| za celý běh: {np.abs(history_hyp).max():.3f}")
print(f"max|chi| na konci (grid by měl být prázdný): {np.abs(history_hyp[-1]).max():.3e}")

In [ ]:
# --- ANIMACE: hyperboloidální řezy ---
every_nth_frame_h = 20

fig, ax = plt.subplots(figsize=(10, 5))
(line,) = ax.plot(rh, history_hyp[0, :], lw=2, color="firebrick")

ax.axvline(S, color="black", lw=1.5, linestyle=":")
ax.text(S, 1.1, "$\\mathcal{I}^+$", ha="center", fontsize=12)

ax.set_xlim(0, S)
ax.set_ylim(-1.1, 1.1)
ax.set_title("Hyperboloidální řezy: odcházející balík $\\chi(t, r) = R\\psi$")
ax.set_xlabel("$r$ (kompaktifikované)")
ax.set_ylabel("$\\chi$")
ax.grid(True, linestyle="--", alpha=0.6)


def update_h(frame):
    line.set_ydata(history_hyp[frame * every_nth_frame_h, :])
    return (line,)


ani_h = FuncAnimation(fig, update_h, frames=len(history_hyp) // every_nth_frame_h, interval=40, blit=True, cache_frame_data=False)

plt.close()

HTML(ani_h.to_jshtml())


In [ ]:
# --- Radiační signál na scri ---
# Tohle je hlavní pointa hyperboloidálního přístupu: vlnový signál čteme
# přímo v bodě r = S (na nulovém nekonečnu), žádná extrakce na konečném poloměru.
t_h = np.arange(len(history_hyp)) * timestep_h
signal = history_hyp[:, -1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_h, signal, lw=2, color="firebrick")
ax.set_xlabel("$t$ (hyperboloidální čas)")
ax.set_ylabel("$\\chi(t, r_{\\mathcal{I}})$")
ax.set_title("Signál na $\\mathcal{I}^+$")
ax.grid(True, linestyle="--", alpha=0.6)
plt.show()

print(f"max signálu na scri: {np.abs(signal).max():.3f} v čase t = {t_h[np.argmax(np.abs(signal))]:.3f}")

**Co je vidět:** balík doletí na $\mathcal{I}^+$ v konečném souřadnicovém čase se zachovanou amplitudou
(u $\chi = R\psi$ se $1/R$ úpadek nekoná) a čistě odteče ven z mřížky. V minkowské části naproti tomu
signál na okraji $R_\mathrm{max}$ čteme v konečné vzdálenosti (a spoléháme na kvalitu Sommerfeldovy
podmínky), zatímco tady je odečet přímo na nulovém nekonečnu a okrajová podmínka není potřeba vůbec.